# FAIR Data Engineering Project
## Preprocessing Notebook

In [24]:
import csv
import re
from typing import *

### Importing Data

In [25]:
INT_REGEX = re.compile("^[0-9]+$")
FLOAT_REGEX = re.compile("^[0-9]+\\.[0-9]+$")


def parse_value(s: str) -> Any:
    if INT_REGEX.match(s):
        return int(s)
    elif FLOAT_REGEX.match(s):
        return float(s)
    elif s == "":
        return None
    else:
        return s


CSVItem = Dict[str, Any]


def import_csv(file_path: str) -> List[CSVItem]:
    with open(file_path, "r", encoding="utf8") as f:
        reader = csv.reader(f)

        column_name = next(reader)
        return list(map(lambda row: {column_name[i]: parse_value(row[i]) for i in range(len(row))}, reader))


def index_by(items: List[CSVItem], key: str | tuple[str, ...]) -> Dict[Any, Dict[str, Any]]:
    get_key: Callable[[CSVItem], Any]
    if isinstance(key, str):
        get_key = lambda item: item[key]
    elif isinstance(key, tuple):
        get_key = lambda item: tuple(item[i] for i in key)

    return {get_key(item): item for item in items}

In [26]:
matches = index_by(import_csv("original_dataset/matches.csv"), "match_id")
player_stats = index_by(import_csv("original_dataset/player_stats.csv"), "player_id")
teams = index_by(import_csv("original_dataset/teams.csv"), "team_id")
match_team_stats = index_by(import_csv("original_dataset/match_team_stats.csv"), ("match_id", "team_id"))
squads_and_players = index_by(import_csv("original_dataset/squads_and_players.csv"), ("player_id", "team_id"))

### Reformatting

In [27]:
def get_player_name(id: int) -> str:
    return player_stats[id]["player_name"]


def get_team_name(id: int) -> str:
    return teams[id]["team_name"]


def get_match_name(id: int) -> str:
    item = matches[id]
    home_id = item["home_team_id"]
    away_id = item["away_team_id"]

    return f"{get_team_name(home_id)} v {get_team_name(away_id)}"


StatItem = Tuple[Any, ...]

#### Player Statistics

In [28]:
PLAYER_STATS_DESCRIPTIONS: Dict[str, str] = {
    "matches_played": "tournament-total number of matches played",
    "matches_started": "tournament-number of matches started",
    "minutes_played": "tournament-total number of minutes played",
    "goals": "tournament-total number of goals scored",
    "assists": "tournament-total number of assists made",
    "shots": "tournament-total number of shots made",
    "shots_on_target": "tournament-total number of shots made on target",
    "yellow_cards": "tournament-total number of yellow cards received",
    "red_cards": "tournament-total number of red cards received",
    "penalty_goals": "tournament-total number of penalty goals scored",
    "own_goals": "tournament-total number of own goals scored",
    "clean_sheets": "tournament-total number of clean sheets",
    "saves": "tournament-total number of saves made",
    "goals_conceded": "tournament-total number of goals conceded",
    "average_rating": "average rating",
}

player_stats_out: List[StatItem] = []

for stats in player_stats.values():
    id = stats["player_id"]
    name = stats["player_name"]

    for label, desc in PLAYER_STATS_DESCRIPTIONS.items():
        amount = stats[label]
        if amount is not None:
            player_stats_out.append((id, name, desc, amount))

SQUADS_AND_PLAYERS_DESCRIPTIONS: Dict[str, str] = {
    "market_value_eur": "estimated market value in Euros",
    "caps": "total international appearances for the national team",
    "height_cm": "height in centimeters",
    "goals": "total international goals scored for their country",
}

for stats in squads_and_players.values():
    name = stats["player_name"]
    id = stats["player_id"]

    for label, desc in SQUADS_AND_PLAYERS_DESCRIPTIONS.items():
        amount = stats[label]
        if amount is not None:
            player_stats_out.append((id, name, desc, amount))

player_stats_out = sorted(player_stats_out)
player_stats_out

[(1, 'José Raúl Rangel', 'estimated market value in Euros', 8933646),
 (1, 'José Raúl Rangel', 'height in centimeters', 190),
 (1,
  'José Raúl Rangel',
  'total international appearances for the national team',
  15),
 (1,
  'José Raúl Rangel',
  'total international goals scored for their country',
  0),
 (1, 'José Raúl Rangel', 'tournament-number of matches started', 5),
 (1, 'José Raúl Rangel', 'tournament-total number of assists made', 0),
 (1, 'José Raúl Rangel', 'tournament-total number of clean sheets', 4),
 (1, 'José Raúl Rangel', 'tournament-total number of goals conceded', 3),
 (1, 'José Raúl Rangel', 'tournament-total number of goals scored', 0),
 (1, 'José Raúl Rangel', 'tournament-total number of matches played', 5),
 (1, 'José Raúl Rangel', 'tournament-total number of minutes played', 438),
 (1, 'José Raúl Rangel', 'tournament-total number of own goals scored', 0),
 (1, 'José Raúl Rangel', 'tournament-total number of penalty goals scored', 0),
 (1, 'José Raúl Rangel', 't

#### Team Statistics

In [29]:
TEAMS_DESCRIPTIONS: Dict[str, str] = {
    "fifa_ranking_pre_tournament": "FIFA ranking before tournament",
    "elo_rating": "ELO rating",
}

team_stats_out: List[StatItem] = []

for stats in teams.values():
    name = stats["team_name"]

    for label, desc in TEAMS_DESCRIPTIONS.items():
        amount = stats[label]
        if amount is not None:
            team_stats_out.append((name, desc, amount))


team_stats_out = sorted(team_stats_out)
team_stats_out

[('Algeria', 'ELO rating', 1760),
 ('Algeria', 'FIFA ranking before tournament', 28),
 ('Argentina', 'ELO rating', 2150),
 ('Argentina', 'FIFA ranking before tournament', 1),
 ('Australia', 'ELO rating', 1790),
 ('Australia', 'FIFA ranking before tournament', 27),
 ('Austria', 'ELO rating', 1830),
 ('Austria', 'FIFA ranking before tournament', 24),
 ('Belgium', 'ELO rating', 1950),
 ('Belgium', 'FIFA ranking before tournament', 9),
 ('Bosnia and Herzegovina', 'ELO rating', 1645),
 ('Bosnia and Herzegovina', 'FIFA ranking before tournament', 64),
 ('Brazil', 'ELO rating', 2030),
 ('Brazil', 'FIFA ranking before tournament', 6),
 ('Cabo Verde', 'ELO rating', 1640),
 ('Cabo Verde', 'FIFA ranking before tournament', 67),
 ('Canada', 'ELO rating', 1795),
 ('Canada', 'FIFA ranking before tournament', 33),
 ('Colombia', 'ELO rating', 1990),
 ('Colombia', 'FIFA ranking before tournament', 13),
 ('Congo DR', 'ELO rating', 1670),
 ('Congo DR', 'FIFA ranking before tournament', 61),
 ('Croatia', 

#### Participation Statistics

In [30]:
MATCH_TEAM_STATS_DESCRIPTIONS: Dict[str, str] = {
    "possession_pct": "ball possession percentage",
    "total_shots": "match-total number of shot attempts",
    "shots_on_target": "match-total number of shots on target",
    "corners": "match-total number of corner kicks taken",
    "fouls": "match-total number of fouls committed",
    "offsides": "match-total number of offsides called against team",
    "saves": "match-total number of saves by the goalkeeper",
}

participation_stats_out: List[StatItem] = []

for stats in match_team_stats.values():
    match_name = get_match_name(stats["match_id"])
    team_name = get_team_name(stats["team_id"])

    for label, desc in MATCH_TEAM_STATS_DESCRIPTIONS.items():
        amount = stats[label]
        if amount is not None:
            participation_stats_out.append((match_name, team_name, desc, amount))

for match in matches.values():
    match_name = get_match_name(match["match_id"])

    home_name = get_team_name(match["home_team_id"])
    home_score = match["home_score"]
    participation_stats_out.append((match_name, home_name, "match-total number of goals scored", home_score))

    away_name = get_team_name(match["away_team_id"])
    away_score = match["away_score"]
    participation_stats_out.append((match_name, away_name, "match-total number of goals scored", away_score))


participation_stats_out = sorted(participation_stats_out)
participation_stats_out

[('Algeria v Austria', 'Algeria', 'ball possession percentage', 50),
 ('Algeria v Austria',
  'Algeria',
  'match-total number of corner kicks taken',
  0),
 ('Algeria v Austria', 'Algeria', 'match-total number of fouls committed', 3),
 ('Algeria v Austria', 'Algeria', 'match-total number of goals scored', 3),
 ('Algeria v Austria',
  'Algeria',
  'match-total number of offsides called against team',
  1),
 ('Algeria v Austria',
  'Algeria',
  'match-total number of saves by the goalkeeper',
  0),
 ('Algeria v Austria', 'Algeria', 'match-total number of shot attempts', 12),
 ('Algeria v Austria', 'Algeria', 'match-total number of shots on target', 5),
 ('Algeria v Austria', 'Austria', 'ball possession percentage', 50),
 ('Algeria v Austria',
  'Austria',
  'match-total number of corner kicks taken',
  3),
 ('Algeria v Austria', 'Austria', 'match-total number of fouls committed', 7),
 ('Algeria v Austria', 'Austria', 'match-total number of goals scored', 3),
 ('Algeria v Austria',
  'Au

## Exporting Data

In [31]:
def export_csv(file_path: str, column_names: Iterable[str], data: Iterable[Iterable[Any]]):
    with open(file_path, "w", encoding="utf8") as f:
        f.write(",".join(column_names) + "\n")

        for r in data:
            f.write(",".join(map(str, r)) + "\n")

        print("exported", file_path)

In [32]:
export_csv("preprocessed/player_stats.csv", ("player_id", "player_name", "stat_desc", "amount"), player_stats_out)
export_csv("preprocessed/teams_stats.csv", ("team_country", "stat_desc", "amount"), team_stats_out)
export_csv("preprocessed/participation_stats.csv", ("match_name", "team_name", "stat_desc", "amount"), participation_stats_out)

exported preprocessed/player_stats.csv
exported preprocessed/teams_stats.csv
exported preprocessed/participation_stats.csv
